# Lab 2 — Ciclo de treino, vazamento e métricas
**Individual · 1,0 ponto na AV1 · 08/10/2026 · horário da atividade no Classroom**

Leia `ENUNCIADO.pdf` antes de preencher. A base comum está incluída; não é necessário usar o banco do projeto.

O notebook executa a preparação e um teste pequeno de ambiente. Nas atividades avaliadas, complete os **TODO** e substitua as mensagens de pendência por resultados. Não há métricas pré-medidas para copiar.

Ordem: preparação → B2 → A → C → D7 → D4–D6 → avaliação final → relatório.

## 0. Ambiente e pasta
No Colab, extraia o ZIP conforme o enunciado e execute `%cd /content/classroom-lab-02` antes da próxima célula. Localmente, abra este notebook na pasta extraída. CPU é suficiente.

In [ ]:
from pathlib import Path
import sys, copy, time, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
from apoio import FEATURES, carregar_base, fixar_semente, criar_modelo
from apoio import treinar_epoca, avaliar_perda, obter_escores

SEED = 42
fixar_semente(SEED)
DISPOSITIVO = 'cpu'
RESULTADOS = Path('resultados')
RESULTADOS.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0])
for pacote in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'torch']:
    print(pacote, importlib.metadata.version(pacote))

### Leitura e conferência da base
`carregar_base` lê o CSV com `;`, confere sua integridade e cria o alvo e os identificadores de controle. As únicas entradas permitidas estão em `FEATURES`. Não aprende média, desvio, pesos ou partições.

Ver as contagens do arquivo não é treinar um modelo. Depois da B2, o teste principal deve permanecer fora das decisões e dos experimentos de erro.

In [ ]:
base = carregar_base()
print(base.head().to_string(index=False))
print('\nLinhas:', len(base), '| características:', len(FEATURES))
print('Classe positiva:', int(base.alvo.sum()), '| proporção:', base.alvo.mean())
print('Perfis distintos:', base.grupo_perfil.nunique())
assert set(FEATURES).isdisjoint({'quality', 'alvo', 'id_linha', 'grupo_perfil'})

### Exemplo mínimo: o que faz uma atualização
O bloco a seguir usa **quatro observações artificiais com duas entradas**, apenas para conferir instalação e formas. Não usa Wine Quality, não mede generalização e não atende às partes avaliadas. Um logit é transformado em escore por sigmoide só ao obter previsões; a perda recebe o logit.

In [ ]:
x_exemplo = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_exemplo = torch.tensor([0., 0., 0., 1.])
exemplo_loader = DataLoader(TensorDataset(x_exemplo, y_exemplo), batch_size=2)
exemplo_modelo = criar_modelo(2).to(DISPOSITIVO)
exemplo_otimizador = torch.optim.Adam(exemplo_modelo.parameters(), lr=0.001)
exemplo_perda = treinar_epoca(exemplo_modelo, exemplo_loader, exemplo_otimizador,
                              nn.BCEWithLogitsLoss(), DISPOSITIVO)
print('Teste de ambiente concluído. Perda de treino artificial:', exemplo_perda)
del exemplo_modelo, exemplo_loader, exemplo_otimizador
fixar_semente(SEED)

## B2. Separe antes de normalizar
Implemente aproximadamente 60/20/20, conservando `grupo_perfil` em uma única partição. Use uma semente declarada. Consulte `GroupShuffleSplit`; proporções de grupos não garantem as mesmas proporções de linhas.

Retorne um dicionário com `treino`, `validacao`, `teste`, cada qual contendo um DataFrame. Mostre cobertura, interseção de linhas e perfis, contagens e classes. Não use IDs como características.

In [ ]:
def particionar_sem_vazar(df, semente=42):
    """TODO B2: retornar as três partições e justificar a regra no README."""
    raise NotImplementedError('Implemente B2 e substitua a atribuição abaixo.')

particoes = None
# Após implementar: particoes = particionar_sem_vazar(base, SEED)
print('B2 pendente: implementar e chamar a função; imprimir a auditoria.')

### B1. Cinco cenários
Responda no README aos cinco cenários do enunciado: avaliações de produtos; turbinas; spam; radiografias; transações. Declare o uso e as hipóteses. Na quinta resposta, discuta tempo e identidade dos clientes. As respostas não precisam de bancos adicionais.

## A1. Dataset próprio
Implemente a classe abaixo com as estatísticas opcionais. Aprenda média/desvio somente no treino e passe-os aos outros conjuntos. Trate desvio zero, preserve o alinhamento e devolva `float32`: X `[11]`, y escalar. Mostre as verificações solicitadas.

In [ ]:
class MeuDataset(Dataset):
    def __init__(self, df, features, media=None, desvio=None):
        """TODO: documentar origem de média/desvio e preparar tensores."""
        raise NotImplementedError('Implemente A1.')

    def __len__(self):
        """TODO: devolver número de observações."""
        raise NotImplementedError

    def __getitem__(self, indice):
        """TODO: devolver X e y alinhados para uma observação."""
        raise NotImplementedError

ds_treino = ds_validacao = ds_teste = None
# Crie os datasets após concluir A1 e B2.
print('A1 pendente: implementar, instanciar e conferir formas/estatísticas.')

## A2. Três DataLoaders
Preencha a tabela de parâmetros no README e crie os carregadores. Na avaliação não descarte observações. `obter_escores` devolve rótulos e escores percorridos juntos, evitando confundir uma ordem de rótulos com outra de previsões.

In [ ]:
loader_treino = loader_validacao = loader_teste = None
# TODO A2: DataLoader para cada Dataset e justificativa dos quatro parâmetros.
print('A2 pendente: criar os carregadores e justificar as opções.')

## A3. Tempo de uma época
Teste os quatro lotes com a rede de apoio **sem BatchNorm**. Controle o estado inicial, novo otimizador, dados e dispositivo; faça aquecimento antes de medir. Percorra o treino inteiro. Use `treinar_epoca`, `time.perf_counter` e, se necessário, sincronização CUDA. Registre repetições e mediana; explique o resultado, inclusive se não for monotônico.

In [ ]:
tamanhos_lote = [1, 8, 64, 512]
tempos = []
# TODO: medir e preencher, por exemplo:
# {'batch_size': 64, 'repeticao': 1, 'segundos': valor_medido, 'passos': valor_medido}
# Salve a tabela e o gráfico em RESULTADOS.
print('A3 pendente: medir quatro tamanhos; não preencher tempos manualmente.')

## A4. BatchNorm com último lote 1
Use um subconjunto de 65 observações do treino e lote 64, ou outra condição equivalente. Confirme o resto. Use `criar_modelo(11, batchnorm=True)`, modo de treino e `drop_last=False`. Capture especificamente o erro esperado, mostre sua mensagem e depois execute a correção que escolheu. Preserve as outras partes do notebook.

In [ ]:
# TODO A4: construir o subconjunto, reproduzir/capturar o erro e corrigir.
# Não deixe uma exceção esperada interromper a execução completa da entrega.
print('A4 pendente: mensagem original e demonstração da correção.')

## C. Dois mecanismos de vazamento
Selecione duas formas; pelo menos uma entre 2/4/7 e uma com dados reais. Um caminho acessível é 1 e 2, usando a base incluída. **Formas 1–7 estão descritas no enunciado.**

Trabalhe apenas com as linhas principais de treino + validação. Crie partições internas `c_...`. Não use a partição principal de teste. Defina o modelo, a métrica e as condições antes de executar o par incorreto/corrigido. Se usar a forma 7, reserve desde o início os quatro conjuntos internos e compare seleções com orçamento igual.

In [ ]:
resultados_c = []
# TODO: criar c_desenvolvimento a partir de particoes['treino'] e ['validacao'].
# TODO: implementar e medir duas formas com seus conjuntos internos.
# Campos sugeridos: forma, fonte, metrica, incorreto, corrigido, diferenca, unidade.
print('C pendente: dois experimentos, medições e explicações próprias.')

## D7. Rede principal com parada antecipada
Construa uma rede nova; não reutilize os pesos ou estatísticas do experimento incorreto da Parte C. A arquitetura de apoio pode ser usada. Complete o laço: treinar, avaliar validação, comparar melhora, copiar o melhor estado, contar paciência, parar e restaurar. Use `copy.deepcopy(modelo.state_dict())` ou tensores clonados para guardar uma cópia independente.

Não forneça o loader de teste à função de treinamento. A perda de validação seleciona a época; o teste não aparece no histórico.

In [ ]:
def treinar_com_parada(modelo, loader_treino, loader_validacao,
                       max_epocas=100, paciencia=10, min_delta=0.0001):
    """TODO D7: devolver histórico, melhor época, época final e melhor perda.
    Ao sair, o modelo deve conter os melhores pesos restaurados.
    """
    raise NotImplementedError('Implemente a lógica de parada e restauração.')

modelo_final = None
historico = None
# TODO: instanciar modelo, chamar a função, mostrar histórico e restauração.
print('D7 pendente: treinar, parar e restaurar antes de produzir escores.')

## D4. Custos como hipóteses explícitas
Descreva a ação de selecionar uma amostra como alta nota. Defina custos positivos de falso positivo e falso negativo, uma unidade e uma justificativa. Não existem custos reais no CSV. Acertos custam zero na simplificação deste laboratório.

In [ ]:
C_FP = None  # TODO: custo e justificativa.
C_FN = None  # TODO: custo e justificativa.
print('D4 pendente: custos e hipóteses no README.')

## D5 e D6. Escolha na validação e fórmula
Use `obter_escores(modelo_final, loader_validacao, DISPOSITIVO)` após a restauração. Compare limiares usando os rótulos da validação. Inclua os casos extremos; registre regra de empate ou resolução de uma grade. Compare com 0,5 e com `C_FP/(C_FP+C_FN)`. Investigue calibração sem transformar sigmoide em garantia de probabilidade.

Salve modelo, normalização, lista de entradas, custos e limiar como escolhas fixadas antes da próxima seção.

In [ ]:
limiar_escolhido = None
decisao_congelada = False
# TODO: busca na validação, comparação de custos/fórmula e registro da decisão.
# Mude decisao_congelada para True somente depois dessas etapas.
print('D5/D6 pendentes: não escolher limiar no teste.')

## D1, D2 e D3. Avaliação final
Só agora obtenha escores no teste com o modelo restaurado e decisão congelada. Avalie as duas regras já fixadas (0,5 e limiar escolhido). O baseline usa a classe majoritária **do treino**. Produza matriz com ordem `[0, 1]`, métricas, prevalência e custos.

AUC-ROC e AUC-PR usam escores. Com `precision_recall_curve`, calcule a área trapezoidal usando `auc(recall, precision)`; se reportar `average_precision_score`, chame-a AP. Na ROC, desenhe a diagonal aleatória. Na PR, desenhe a prevalência do teste.

Não use o resultado do teste para voltar à busca. Relate os limites observados.

In [ ]:
avaliacao_final = None
if not decisao_congelada:
    print('Avaliação final pendente: concluir o desenvolvimento e congelar a decisão.')
else:
    # TODO: obter rótulos/escores do teste uma vez, calcular e salvar tabelas/curvas.
    print('TODO D1–D3: implementar a avaliação das regras já fixadas.')

## D8. Recomendação final
Escreva no README o parágrafo para quem decide: ação, limiar, evidência final, custos hipotéticos, riscos e limitações. Cite o conjunto avaliado. Não reivindique desempenho em novos produtores, regiões ou datas não observados.

## Conferência da entrega
Reinicie o kernel e execute tudo. O notebook preenchido deve mostrar as saídas de todas as partes; pendências não são resultados. Preencha o README e o arquivo obrigatório de uso de IA. Inclua as referências e registre alterações no apoio.

In [ ]:
pendencias = []
for nome, valor in [('particoes', particoes), ('ds_treino', ds_treino),
    ('loader_treino', loader_treino), ('historico', historico),
    ('limiar_escolhido', limiar_escolhido), ('avaliacao_final', avaliacao_final)]:
    if valor is None:
        pendencias.append(nome)
if not tempos: pendencias.append('tempos A3')
if len(resultados_c) < 2: pendencias.append('dois resultados C')
print('Pendências estruturais:', pendencias)
print('Esta checagem não corrige a análise nem substitui o checklist do enunciado.')